# ```Imports```

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F
# Torch will handle all the established maths behind the curtain fro us, lot of calculus and tensors.
# let's set up owr devices now;
device='cuda' if torch.cuda.is_available() else 'cpu'

# hyper parameters
block_size=8
batch_size=4
# The avobe parameters are very very important for training.
# We will see how important they are when we scale our model and the parameters will keep getting more and more complex.


# SideEffects
print(device)
# cuda is available
!nvidia-smi

cuda
Sat Nov 21 17:27:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.99.02              Driver Version: 595.99.02      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060 Ti     Off |   00000000:01:00.0  On |                  N/A |
|  0%   44C    P8             21W /  240W |     620MiB /   8192MiB |      9%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+

# ```Dataset```
Let's download a small dataset from ```Project Guttenberg```(https://www.gutenberg.org)
We are going to train a ```transformer``` or a ```bigram language model``` with our preparred dataset

In [3]:
# We are opening the text in read mode with encoding 'utf-8'
with open('wizard_of_oz.txt','r',encoding='utf-8') as f:
    text=f.read()
# Plain text is a list of strings or char s
print(len(text))
chars=sorted(set(text))
# Sorted is not for sorting but for creating a list from a set
vocab_size=len(chars)
print(vocabulary_size)
print(chars)

232308


NameError: name 'vocabulary_size' is not defined

In [ ]:
# Now we are are going to use tokenizers with generative for loops and custom functions;
string_to_int={ch:i for i,ch in enumerate(chars)}
int_to_string={i:ch for i,ch in enumerate(chars)}
encode=lambda s:[string_to_int[c] for c in s]
#decode=lambda l:''.join([chars[int(*j)] for j in l])
decode=lambda l:''.join([int_to_string[j] for j in l])
# All the errors because I had an extra pair of []s

In [ ]:
# same as the above lambda
def encode(s):
    encode_list=[]
    for c in s:
        encode_list.append(string_to_int[c])
    return encode_list
def decode(l):
    decode_list=[]
    for j in l:
        decode_list.append(int_to_string[j])
    decoded_text=''.join(decode_list)
    return decoded_text

In [ ]:
# Let's test the lambdas
encoded_hello=encode('hello')
# So now we have a conversion from characters to integers so let's decode it back into our feature
decoded_hello=decode(encoded_hello)
encoded_hello,decoded_hello
# Is this the so called embedding?

# ```Tokenizers```:
Right now we have been using ```character level``` tokenizer, it takes each ```character``` and converts them into an ```integer```.
We have a ```small vocabulary``` but a ```large number of tokens```(232308 characters) that we have converted into ints.
If we had a ```word level tokenizer``` meaning all the possible ```meaningful arrangements``` of group of characters.
So the ```biggest real world word``` having ```r number of characters``` will belong to ```{{nPr+np(r-1)+.....+nP(r-r)}and{all the words in the dictionary}}``` while n=80 according to above len('tf-8' encoding)
So Ton of more tokens, maybe millions or billions of tokens
Ususally we will be ```between``` character level or word level tokenizer
# ```Efficiency``` in Dataset
In the context of language model it is very important that we are efficient with our data.
Just having a giant stream might not work the best

In [ ]:
# let's unleash the torch! khuuuuuuu
# Our guy wants to use torch.long so int64; he thinks it's because we will be dealing with a super long sequence of tensors, so bigger chunks in ram assignment the better
data=torch.tensor(encode(text),dtype=torch.long)
data.ndim,data[:100]
# It has only one dimensions as bunch of integers inside one set of []s

# Let's talk about training and validation ```splits```
We need to ```train``` on a part of the ```corpus``` and keep remaining parts for ```validation``` or testing
If we train on the ```whole corpus```, it will ```memorize``` everything; that spoils the intelligence.
That testing part is what makes the ```adaptive nodes``` inside the black box

In [ ]:
# Let's get the split at around 80% of the data and n gives us exact iteror index at the exact slice of 0.8*232308 from the start
n=int(0.8*len(data))
train_data=data[:n] # 80% of the data
val_data=data[n:] # remaing 20% of the data
# So the train_data consists of the inputs and val_data consists of the y_i s?

def get_batch(split):
    # Reassigning data;
    data=train_data if split=='train'else val_data # Is train some kind of flag?
    # Two exclusive stream of data selection
    ix=torch.randint(len(data)-block_size,(batch_size,)) # The second argument takes in a touple and we have a V so 4D vector, for stacking 8 members in each of the 4 rows later
    # len(data)-block_size is the random range of torch.randint() formation, meaning each elements ranging from this huge number.
    # We are taking the entirity of either train_data or val_data minus the block size so that overlapping doesn't happen as we are already dealing with some batch of 8 chars somewhere in data
    # So ix is a 4D vector of huge nundom numbers!
    #[data[i:i+block_size] for i in ix] # Just takiing horizontal slices of block size from the indices of encoding from ix in each iteration
    # data[x:y] refers (from either train_data or val_data) consecutive members from x to y excluding y, so now let's torch.stack the 8 members of the above list comprehension
    print(ix)
    inputs=torch.stack([data[i:i+block_size] for i in ix])
    # Now we will slide by one char so;
    targets=torch.stack([data[i+1:i+block_size+1] for i in ix])
    # One more thing, let's send the inputs and outputs to device
    inputs,targets=inputs.to(device),targets.to(device) # WE ARE USING device='cuda:0' for these tensors!!
    # These batches will not be processed sequentially on cpy rather parallelly on gpu!
    # That's how we push any piece of data into gpu!
    return inputs,targets

x,y=get_batch('train')
# Train is just a flag. We gave flag that we are in the training zone!
x,y

# ```Bigram Bi Prefix Means 2```
like the ```in between``` aproach in animation, but different in a sense that if it has a character h then an e after that will have ```bigger probabilty``` than l given the  real world usecase of ```meaningful word making```.
As ```two letters are involved``` and thus ```weight```(of the second letter) is determined based on the ```position of two letters one after another``` it's called a bigram model.
Similar models we can generate for our ```steps lesson from mewstro``` where the second note(```target```) is weighted by the context of the first note(```context```).

# ```Block Size```
Instead of taking the whole corpus size we are going to take ```small snippets```.
We are going to ```make some predictions``` and we are going to ```decide some targets``` out of that.
Let's take a ```block size``` of 5(probably a trade off between the compression and the accuracy of the predictions) and observe two ```consecutive interlocking blocks``` in a row.

1. [...[5, 67, 21, 58, 40],35 ...]
2. [... 5,[67, 21, 58, 40, 35]...]

So the next block is ```slided over``` the other at the ```interlocked zone``` of a text_list==[67, 21, 58, 40]

In [ ]:
# To implement the blocks pythonically we will start the snippet from the slice where the first_interlocking_snippet==encoded_text[:block_size] and second_interlocking_snippet==encoded_text[1:block_size+1] 
#block_size=8
x=train_data[:block_size] # x is a list
y=train_data[1:block_size+1] # y is a list
# Both x,y are the interlocking slices from the training data which starts from data[0]
for t in range(block_size): # iteration number = block_size && t=0, t+=1 on each iteration upto t=block_size-1
    # Context is the elements inside the list;
    context= x[:t+1] # assignment of context, meaning the first of the two interlocking (list|chunk|block)-populating like the sheeps from x[0:1] to x[0:block_size]
    # target is the element outside the list but the index=0 of the second (list|chunk|block)
    target=y[t]
    # We are not appending so all the assignments are being reassignments withe modified indexes on each iteration
    print(f'context=={context} gives target=={target}')

# This Isn't Scalable Yet!
This is ```sequencial```; sequencial is another way to describe ```what a cpu does as opposed to gpus```.
CPUs can do a lot of complex tasks very quickly but it only happens sequencially.
On the other hands, GPUs we can do a lot of simplersmaller computations very very quickly and in parallel.
The processors of gpus are much inferierior as units but they come in large numbers.
So what we can do is we can take each of these little blocks and then we can stack them to push these to GPUs to scale our training, a lot.

# Batches as a Hyper Parameter
For illustration let's take 4 blocks and stack them on top of one another;
```
[... ...]
[... ...]
[... ...]
[... ...]
```
We can represent this new block of 4 blocks as a new hyper parameter called batch size.
It tells us how many of these sequences or block of loops we can process in parallel. (So a 2d tensor?)


```
batch_size=[[... ...]
            [... ...]
            [... ...]
            [... ...]]
```                   

So we just went over how we get batches or rather how we can utilize this second layer of representation to accelerate the training process using the parallel processing muscle of the GPUs.
Let's take our block_size variable up at the import cell where other global variables like batch_size will be defined.


# Time For Neural Network
## Loss
Let's say we have 80 chars in our vocabulary, and we start our model with random weights and biases, meaning the first forward feed. In this state we have a $\frac{1}{80}$ chance to predict the next token correctly and to translate $-ln(\frac{1}{80})$% into is what loss actually looks like.
So pretty much the point is to minimize the loss.
While updating $\vec{\theta}$ according to $\vec{\theta}'=\vec{\theta}-\alpha\vec{\nabla_{w,b}C}$ the $\alpha$, the so called learning rate, is what enables us to have a proper float value to subtract|add with the analogous float value of $\vec{\theta}$.
We will keep adjusting the parameters so that our gradients change it's course towards the minima on each iteration.
Gradient descent optimizer.
Let's browse through torch.optim to explore other optimizers.
## Optimizers
We are going to use AdamW to implement AdamW algorythm.
Adam is a popular optimization algorythm that combine ideas of momentum and it uses a moving outage of it's gradient and the squared value to adapt the learning rate $\alpha$ of each parameter.
## Learning Rate
- Higher learning rate is like taking long steps. Let's consider $f(x)=sin(x)$ where taking big steps at the steep $f'(x)$ descent might suddenly bring back into gradient ascend where cost is even higher, and it happens faster.
- Let's consider $\vec{\theta}'=\vec{\theta}-\alpha\vec{\nabla_{w,b}C}$ which involves a plethora of vectors with varying magnitudes and impacts as they contain both $\vec{b^l}$ biases and $W_{jk}^l$ weights where wights have varying costs depending on $\vec{a}_k^{l-1}$ and we end up with a non complanar surface representing the big $\vec{\theta}$ as a function.
- In this intricate situation subtle learning rate will give us most efficient $\vec{\theta}$.
The non coplanar surface can be a plane where biases are on x axis, weights on 2 axis and cost on Z axis. Then we will do partial gradients as snippets on bias axis or weights axis.
- AdamW is another layer of modification into Adam algoryth. It has weight decay, it generalizes the parameters more, the learning rate specially. It's the sweet spot in between high level and low level performance.
- The weight significance will actually shrink as it flattens out and make sure certain parameters in the network, of weight matrices isn't affcting the output of the model drastically in both positive or negative direction. There can be some lucky high performances and we need to rely on the solid rule of big numbers kinda things.
- 

In [ ]:
class BigramLanguageModel(nn.Module):
    # It will be a subclass of nn.Module and inherit the superclass variables and methods
    # If we use nn.Linear inside this class that will mean the nn.Linear paeameters are learnable; the weight matrix can be changed with gradient descent.
    def __init__(self,vocab_size):
        super().__init__()
        self.token_embedding_table=nn.Embedding(vocab_size,vocab_size) # So just 80 from 80; comparing with the gpt example of using_cuda.ipynb
        # It's just creating an attribute .token_embedding_table where some attributes are from super class

    def forward(self,index,targets=None): # So we can exclude targets and wverything will stay just fine
        # This is a class method where we define everythin feed forward.
        # We are making custom function instead of using what it has by default because it is a good practice to know what is happening behind the scene in our model.
        # We wanna know exactly what's going on so that we can be more flexible and creative.
        # If we are using weird data and defaults are not used to it we are going to get bugs from that. 
        logits=self.token_embedding_table(index) # Pulling the attribute from the init block which is just a nn.Embedding() ready and hungry for indices.
        # The nn.Embedding() is pretty much just a look up table, we are gonna have a giant sort of grid of what the predictions are gonna look like.
        if targets is None:
            loss=None
        else:
            # We are gonna use a function called .view which is gonna help us sorta reshape how logit looks like in human form.
            # B -> batch; T - Time Dimension, through the generation for some we do know and some we don't know yet; C-> Channel stands for vocab_size 
            B,T,C=logits.shape
            # Ar above, we just assined the touple from logit.shape into B,Tand C
            # So basically logits will have a table of index amount of rows and vocab_size amount of columns.
            # The table has size=(index,vocab_size).
            # Below we are reducing logit.shape from 3D to 2D;
            logits=logits.view(B*T,C) # B and T are usually blended together because they are not as important here
            # As long as the logits and the targets have same (B,T) we shall be fine.
            # Let's now initialize our targets, it's just 1D;
            targets=targets.view(B*C) # Probably targets are supposed to have one dimension of elements less.
            # Why do we have to reshape the logits and targets? well the answere falls into what does cross_entropy() expects the shapes to be of its arguments.
            loss=F.cross_entropy(logits,targets) # We are bringining functional's cross_entropy() method which takes in two args.
            # We provided the cross_entropy() with logits and targets, two .view() methods of our nn.Embedding() objects. 
            # It is very important to stay well informed with what and how each pytorch function expects the arguments to be.
        return logits,loss
        # Logits are a kind of probability distribution, a more sophisticated version of normalization; it's kinda the probability distribution of all  the 80 other letters in the context of one letter.

    # Now Let's define generate which returns index;
    def generate(self,index,max_new_tokens:list):
        # index is the (B,T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions; probably index will be a class variable, we are creating everything out of indexes. WHAT IS INDEXES!!!!!
            logits,loss=self.forward(index) # logits and losses are being generated exactly as defined in forward() block
            # Focus only on the last time step
            logits=logits[:,-1,:] # Becomes (B,C)
            # Apply functional's softmax to get probabilities. Goal: softer nudges on our random logits
            probs=F.softmax(logits,dim=-1) # (B,C) to (B,1)
            # Sample from the distribution, again index! next_index!
            index_next=torch.multinomial(probs, num_samples=1) # Plugging in (B,1)
            # Append sampled index to the running sequence, I KNOW IT, IT MAKES LONGER TENSORS CONCATINATING SMALLER ONES LIKE LISTS!
            index=torch.cat((index,index_next),dim=1) # dim 1 because we need to simplify into a long vector for the simplicity of the algorythm or programming
            # The game of indexes like game of thrones.
        return index
        # Finally returning some modified version of index

# Let's create the model object in the main;
model=BigramLanguageModel(vocab_size) # Just putting the init variable to execute the construction, and it's pulling the nn materials from the superclass nn.Module
m=model.to(device) # Just abreviating and sending the model wherre it belongs!
# Just sending our fine tuned indexes into the decode while unleashing the class methode generate where putting a zeros tensor as context.
context=torch.zeros((1,1),dtype=torch.long,device=device) # Single input of zero; [o] as context and of course cuda!
# max_new_tokens is the kwarg for iterations in generate for loop, may be this is the bulk iteration phase.
generated_chars=decode(m.generate(context,max_new_tokens=500)[0].tolist()) # Decode just takes in indexes, and our model determines how intelligent the INDEXES ARE!
# torch.int64 because it's a game of indexes!
print(generated_chars)
'''
SO PROUD TO GENERATE THIS!
(3,0m-nRWbI,xA3kf)s6w2N_FTio80*hpa
6IACTFzUWSP2t8LhsFe 8hUR?R0*Z5CZa7jl6F;TI5[zURVbn3t.R_;NMOlCpirGD!;o(Fk 
5[UBmm.Mrb9_Cpp7OK9oY7Cp&Vw ,iPl!K)G n8ft81FHsnEiJ.iws4kH3ko1iB&ge(;X[D3T;Q_ADS4":G;Z55tmccH"YGN44"?t&?VG!aG,VBTj'?7I,-R9*c.:x!yEA
ggB?c'-]Pp!G)gbW:O447xOAup
5[oTSQpi14,seaA&!("8dcsFyRV&gNYS&_G"Tblv*nrT&VBoru:R,kG"eB1sJVovKIcRPJJ!ajgnxDt.]L9nf["l]LT4,tu"&!1VbSN?Ga!8C&to
R, "x DRPTR?YabtCFg3,Hl-Pl13k8:;I
q-CUdZ0wHdFr"r-LE&'':ok2(8K6tW"25tvOAH)gn?cQZsbwsTb9!_gKdcTY-Ey:8;7: tva"e,RVL3,TU_rj-VH)
'''
# HAHAHAHAHAHAHAHA!
        

# Why SHAPE and VIEW Methods
According to the documentation the method expects a shape as $(C),(N,C)or(N,C,d_1,d_2,...,d_K)$  with $K\ge 1$ where;
$$C=\text{number of classes}$$
$$N=\text{batch size}$$
- For our case batch size is denoted by B So we need to redistribute our acquired shape (B,T,C) from nn.Embedding() into (B,C) for logits and (C) for targets. We could just retype it by hand but .view() gives us this scope of reallocation of dimensions.
- Also for our case we are implementing the T time by multiplying it with B to get a more efficient version of B and for targets we are implementing the Batch size B by multiplying it with C to get a more efficient version of C.
## SHAPE And VIEW Methods Help Reshapeing The Existing Tensors

In [ ]:
# Example of .shape and .view;
a=torch.rand(2,3,5)
x,y,z=a.shape # It will assign a touple to our variable touple
print(a,a.view(x*y,z),a.view(x,y*z),a.view(x*y*z))
# View is used to redistribute the elements into all possible sizes, but total item numbers 30 has to be equal in each cases.

# Breaking Down the `logits[:,-1,:]` Range In Each Dimension

The variable `logits` usually represents a 3D tensor with the shape `(batch_size, sequence_length, vocabulary_size)`. Here is how the slice `[:, -1, :]` targets each dimension:

- `:` (First Dimension - Batch Size): The colon selects all batches in the current forward pass.
- `-1` (Second Dimension - Sequence Length): Python uses negative indexing to count from the back. `-1` selects only the `i=-1` (the most recently generated or inputted token) in the text sequence. 
- `:` (Third Dimension - Vocabulary Size): The final colon retains the entire vocabulary distribution (raw scores for every possible word/token) for that last position.

If We had [:,:,:] it would select all. For each in betweem comma we can apply the `iterable[j:k]` aproach meaning the logits being a 3D tensor can show 3 separate indexing option like `logits[i:j,k:l,m:n]`.